# 🕵️ Dia 86 — Forense: fundindo três logs em uma linha do tempo

**Investigador Cyber 90 Dias** · Fase 7 · Rank Jutsu `S`

> Baseado no *Day 86* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-7-capstone-career/day-86.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).
> **Material sintético.** O *case packet* (Orrin Valley / Castellan) e a Operação Isca são ficção; domínios `.example`, IPs de documentação.

---

### 🎬 Onde paramos

A linha do tempo da Serra Azul ficou absurda: o cooperado V1 contestou o Pix **antes** de o Pix sair. Rafa: *"Ou o V1 tem bola de cristal, ou o relógio está errado. Não aposta. Ancora."*

Neste notebook você vai:

1. **Achar as âncoras automaticamente** (POST de login no proxy × `UserLoggedIn` na auditoria).
2. **Ver a armadilha de 3 de março**: o mesmo horário convertido com -4 e com -5.
3. **Rodar o script do original** e conferir os 80 eventos.
4. **Montar a tabela de sessões** e jogar o quiz de atribuição.
5. **Testar a evidência negativa** do envio de 11 de março.
6. **Repetir tudo com os três relógios da Operação Isca.**

🐍 Só biblioteca padrão do Python. Funciona offline: se a rede falhar, o notebook usa a cópia embutida dos arquivos. Rode as células de cima para baixo (Shift+Enter).

## 📦 O pacote do caso (plano A: GitHub; plano B: cópia embutida)

A próxima célula guarda uma **cópia embutida** dos arquivos do *case packet* que este dia usa. Ela é o plano B: se você estiver sem internet, ou se o GitHub estiver fora, o notebook continua funcionando.

A célula seguinte tenta o plano A: baixar cada arquivo do repositório original com `urllib`. Se baixar, compara com a cópia embutida pelo SHA-256 e avisa se o original mudou desde que este notebook foi gerado (é exatamente o raciocínio de custódia do Dia 83).

In [ ]:
# Cópia embutida (plano B). Cada valor é o texto exato do arquivo do case packet.
# Não edite: se mudar um caractere, o SHA-256 muda e a custódia quebra.
EMBUTIDO = {
"02-suspicious-email.eml": "X-Case-Packet-Notice: SYNTHETIC TRAINING DATA. Fictional organizations, people, domains (RFC 2606 .example) and IPs (RFC 5737). Do not look up or contact anything in this file.\nReturn-Path: <bounce-7731@castellan-hardwood.example>\nReceived: from gw01.orrinvalley.example (gw01.orrinvalley.example [192.0.2.25])\n\tby mbx01.orrinvalley.example with ESMTPS id 4f1c9a2e\n\tfor <jpike@orrinvalley.example>; Tue, 10 Mar 2026 14:02:39 +0000\nReceived: from mail.castellan-hardwood.example (vps-47.pinecrate.example [203.0.113.47])\n\tby gw01.orrinvalley.example with ESMTPS id 8b3e0d17\n\tfor <jpike@orrinvalley.example>; Tue, 10 Mar 2026 14:02:37 +0000\nReceived: from [10.8.0.6] (unknown [198.51.100.23])\n\t(Authenticated sender: ar@castellan-hardwood.example)\n\tby mail.castellan-hardwood.example (Postfix) with ESMTPSA id 2C4A1E03\n\tfor <jpike@orrinvalley.example>; Tue, 10 Mar 2026 14:02:35 +0000\nAuthentication-Results: gw01.orrinvalley.example;\n\tspf=pass (gw01.orrinvalley.example: domain of bounce-7731@castellan-hardwood.example designates 203.0.113.47 as permitted sender) smtp.mailfrom=castellan-hardwood.example;\n\tdkim=none (message not signed);\n\tdmarc=none action=none header.from=castellan-hardwood.example\nX-OVM-Gateway-Verdict: clean; score=3.1; threshold=5.0; tags=EXTERNAL\nDate: Tue, 10 Mar 2026 10:02:31 -0400\nFrom: \"Dana Whitlock | Castellan Hardwood Supply\" <dana.whitlock@castellan-hardwood.example>\nReply-To: <dwhitlock.castellan@freemail.example>\nTo: Jordan Pike <jpike@orrinvalley.example>\nSubject: [EXTERNAL] RE: Invoice CHS-20417 - updated remittance details\nMessage-ID: <20260310140231.2C4A1E03@mail.castellan-hardwood.example>\nIn-Reply-To: <20260303144702.GA11873@mail.castellanhardwood.example>\nReferences: <20260303144702.GA11873@mail.castellanhardwood.example>\nUser-Agent: Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Thunderbird/115.8.0\nMIME-Version: 1.0\nContent-Type: multipart/alternative; boundary=\"------------b7Qx2LmR0v8aZ3\"\n\nThis is a multi-part message in MIME format.\n--------------b7Qx2LmR0v8aZ3\nContent-Type: text/plain; charset=UTF-8\nContent-Transfer-Encoding: 7bit\n\nHi Jordan,\n\nFollowing up on the invoice below. As part of our annual audit we have\nmoved our receivables to a new account, effective this week. Please use\nthe updated remittance details for CHS-20417 and for all future payments.\nThe signed bank letter and the vendor update form are on our secure\nportal:\n\nView remittance form (secure): https://castellanhardwood.example/remittance\n\nOur office phones are being migrated to a new system this week, so email\nis the quickest way to reach me. We would appreciate it if CHS-20417 could\ngo out in Friday's payment run, as the old account closes on the 16th.\n\nThanks so much,\n\nDana Whitlock\nAccounts Receivable Manager\nCastellan Hardwood Supply\nTel. 555-0148 | ar@castellan-hardwood.example\n\nOn Tue, Mar 3, 2026 at 9:47 AM Dana Whitlock <dwhitlock@castellanhardwood.example> wrote:\n> Good morning,\n>\n> Please find attached invoice CHS-20417 for the February white oak and\n> hard maple order (PO 7719), total $48,612.50, net 10.\n>\n> Thank you for your business.\n>\n> Dana Whitlock\n> Accounts Receivable Manager\n> Castellan Hardwood Supply\n> Tel. 555-0112 | accounts@castellanhardwood.example\n\n--------------b7Qx2LmR0v8aZ3\nContent-Type: text/html; charset=UTF-8\nContent-Transfer-Encoding: 7bit\n\n<html><body style=\"font-family:Calibri,Arial,sans-serif;font-size:11pt\">\n<p>Hi Jordan,</p>\n<p>Following up on the invoice below. As part of our annual audit we have moved our receivables to a new account, effective this week. Please use the updated remittance details for CHS-20417 and for all future payments. The signed bank letter and the vendor update form are on our secure portal:</p>\n<p><b>View remittance form (secure):</b> <a href=\"https://cstl-docshare.example/view/r?u=b3JyaW52YWxsZXkuZXhhbXBsZQ%3D%3D&amp;t=anBpa2U%3D\">https://castellanhardwood.example/remittance</a></p>\n<p>Our office phones are being migrated to a new system this week, so email is the quickest way to reach me. We would appreciate it if CHS-20417 could go out in Friday's payment run, as the old account closes on the 16th.</p>\n<p>Thanks so much,</p>\n<p>Dana Whitlock<br>Accounts Receivable Manager<br>Castellan Hardwood Supply<br>Tel. 555-0148 | ar@castellan-hardwood.example</p>\n<img src=\"https://castellanhardwood.example/img/logo.png\" width=\"160\" alt=\"Castellan Hardwood Supply\">\n<div style=\"border-left:2px solid #ccc;padding-left:8px;color:#555\">\n<p>On Tue, Mar 3, 2026 at 9:47 AM Dana Whitlock &lt;dwhitlock@castellanhardwood.example&gt; wrote:</p>\n<p>Good morning,</p>\n<p>Please find attached invoice CHS-20417 for the February white oak and hard maple order (PO 7719), total $48,612.50, net 10.</p>\n<p>Thank you for your business.</p>\n<p>Dana Whitlock<br>Accounts Receivable Manager<br>Castellan Hardwood Supply<br>Tel. 555-0112 | accounts@castellanhardwood.example</p>\n</div>\n</body></html>\n\n--------------b7Qx2LmR0v8aZ3--\n",
"logs/mail-trace.csv": "# SYNTHETIC TRAINING DATA. Fictional organizations, people, domains (RFC 2606 .example) and IPs (RFC 5737).\n# Orrin Valley Millworks mail gateway (gw01) message trace export, 2026-03-01 to 2026-03-17.\n# Filter: sender, recipient or subject contains \"castellan\", \"jpike\" or \"mbeaulieu\". Times are UTC.\nreceived_utc,direction,message_id,sender,recipients,subject,sender_ip,spf,dkim,dmarc,verdict,delivery_action,attachment_name,attachment_sha256\n2026-03-02T15:12:40Z,inbound,<n.8812.20260302@lists.sawdustjournal.example>,news@sawdustjournal.example,jpike@orrinvalley.example,[EXTERNAL] Sawdust Journal - March issue,198.51.100.90,pass,pass,pass,clean,delivered,,\n2026-03-03T14:47:04Z,inbound,<20260303144702.GA11873@mail.castellanhardwood.example>,dwhitlock@castellanhardwood.example,ap@orrinvalley.example;jpike@orrinvalley.example,[EXTERNAL] Invoice CHS-20417,198.51.100.180,pass,pass,pass,clean,delivered,CHS-20417.pdf,a3c91f0e7b24d85c6e10f9a2b7d43c8e5f6a1b09d2e7c4f38a5b6d0e1f2c7a934\n2026-03-04T19:30:11Z,outbound,<7c21e0a9f4b3@mbx01.orrinvalley.example>,jpike@orrinvalley.example,orders@castellanhardwood.example,PO 7744 - walnut 8/4,192.0.2.25,,,,,relayed,PO-7744.pdf,e8b04d2c9a71f35e6c0b8d49a2f1e7c35d6a90b1c4e2f87a3d5b9c0e6f1a2d48\n2026-03-09T13:40:22Z,internal,<3e9b17c0d5a2@mbx01.orrinvalley.example>,mbeaulieu@orrinvalley.example,jpike@orrinvalley.example,Friday run list,,,,,,delivered,,\n2026-03-10T14:02:37Z,inbound,<20260310140231.2C4A1E03@mail.castellan-hardwood.example>,dana.whitlock@castellan-hardwood.example,jpike@orrinvalley.example,[EXTERNAL] RE: Invoice CHS-20417 - updated remittance details,203.0.113.47,pass,none,none,clean,delivered,,\n2026-03-11T13:21:06Z,outbound,<5b2e90d14c7a@mbx01.orrinvalley.example>,jpike@orrinvalley.example,orders@castellanhardwood.example,PO 7751 - hard maple 5/4,192.0.2.25,,,,,relayed,PO-7751.pdf,0f7d3a9c2e15b84d6a09c1e7f3b25d80a4c6e9f1b7d02a53c8e4f6b19d0a7c25\n2026-03-11T13:52:45Z,internal,<a81f3c07e2d94b6e@mbx01.orrinvalley.example>,jpike@orrinvalley.example,mbeaulieu@orrinvalley.example,Castellan banking update - vendor change for Friday run,,,,,,delivered,Castellan_Bank_Letter_2026.pdf,5d0c7a1e9b43f26c8e7a0d51b9f4c3e2a6d8b17f04e9c5a2d3b6f8e1c7a9042b\n2026-03-11T15:06:13Z,internal,<c40d8e2b71f6a953@mbx01.orrinvalley.example>,mbeaulieu@orrinvalley.example,jpike@orrinvalley.example,RE: Castellan banking update - vendor change for Friday run,,,,,,delivered,,\n2026-03-12T16:05:40Z,inbound,<20260312160538.GB20417@mail.castellanhardwood.example>,dwhitlock@castellanhardwood.example,ap@orrinvalley.example;jpike@orrinvalley.example,[EXTERNAL] Reminder: CHS-20417 due 13 March,198.51.100.180,pass,pass,pass,clean,delivered,,\n2026-03-12T16:40:03Z,inbound,<20260312163958.8E71B2C4@mail.castellan-hardwood.example>,dana.whitlock@castellan-hardwood.example,jpike@orrinvalley.example,[EXTERNAL] RE: Invoice CHS-20417 - updated remittance details,203.0.113.47,pass,none,none,clean,delivered,,\n2026-03-13T19:02:09Z,outbound,<erp.000731.20260313@orrinvalley.example>,erp-noreply@orrinvalley.example,ar@castellan-hardwood.example,Remittance advice - Orrin Valley Millworks - payment 000731,192.0.2.25,,,,,relayed,,\n2026-03-16T14:20:55Z,inbound,<20260316142051.GC20417@mail.castellanhardwood.example>,dwhitlock@castellanhardwood.example,ap@orrinvalley.example;jpike@orrinvalley.example,[EXTERNAL] CHS-20417 not received - please call me,198.51.100.180,pass,pass,pass,clean,delivered,,\n",
"logs/mailbox-audit.csv": "# SYNTHETIC TRAINING DATA. Fictional organizations, people, domains (RFC 2606 .example) and IPs (RFC 5737).\n# Orrin Valley Millworks cloud mailbox and sign-in audit export (simplified format).\n# Scope: all events for jpike and mbeaulieu, 2026-03-09T00:00Z to 2026-03-17T12:00Z; admin actions targeting those mailboxes;\n# plus failed sign-ins tenant-wide from 203.0.113.200. Times are UTC. Exported 2026-03-17 by the IT contractor.\ntimestamp_utc,actor,target_mailbox,operation,result,client_ip,user_agent,session_id,details\n2026-03-09T12:58:10Z,mbeaulieu,mbeaulieu,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-3a9e41,AuthMethod=Password+AuthenticatorApp; MFA=Satisfied\n2026-03-09T13:02:31Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-12c9d0,AuthMethod=Password; MFA=NotRegistered\n2026-03-09T21:47:02Z,jpike,jpike,UserLoggedIn,Success,198.51.100.77,\"Mozilla/5.0 (iPhone; CPU iPhone OS 17_3 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148\",s-58b2e7,AuthMethod=Password; MFA=NotRegistered\n2026-03-10T03:11:40Z,jpike,jpike,UserLoginFailed,InvalidPassword,203.0.113.200,python-requests/2.31.0,,AuthMethod=Password\n2026-03-10T03:11:52Z,mbeaulieu,mbeaulieu,UserLoginFailed,InvalidPassword,203.0.113.200,python-requests/2.31.0,,AuthMethod=Password\n2026-03-10T03:12:05Z,otran,otran,UserLoginFailed,InvalidPassword,203.0.113.200,python-requests/2.31.0,,AuthMethod=Password\n2026-03-10T03:12:19Z,panand,panand,UserLoginFailed,InvalidPassword,203.0.113.200,python-requests/2.31.0,,AuthMethod=Password\n2026-03-10T03:12:33Z,reception,reception,UserLoginFailed,InvalidPassword,203.0.113.200,python-requests/2.31.0,,AuthMethod=Password\n2026-03-10T03:13:05Z,sales,sales,UserLoginFailed,InvalidPassword,203.0.113.200,python-requests/2.31.0,,AuthMethod=Password\n2026-03-10T13:05:14Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-12f4a8,AuthMethod=Password; MFA=NotRegistered\n2026-03-10T13:08:40Z,mbeaulieu,mbeaulieu,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-3b07c2,AuthMethod=Password+AuthenticatorApp; MFA=Satisfied\n2026-03-10T14:31:12Z,jpike,jpike,UserLoggedIn,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f3a61,AuthMethod=Password; MFA=NotRegistered\n2026-03-10T14:36:50Z,jpike,jpike,New-InboxRule,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f3a61,Name=.; SubjectOrBodyContainsWords=Castellan;remittance;banking;CHS-; MoveToFolder=RSS Subscriptions; MarkAsRead=True; StopProcessingRules=True\n2026-03-10T14:38:02Z,jpike,jpike,MailItemsAccessed,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f3a61,AccessType=Sync; Folder=Inbox; ItemCount=212\n2026-03-10T14:41:27Z,jpike,jpike,MailItemsAccessed,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f3a61,AccessType=Bind; Folder=Inbox; Subject=[EXTERNAL] Invoice CHS-20417\n2026-03-10T14:44:09Z,jpike,jpike,MailItemsAccessed,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f3a61,AccessType=Sync; Folder=Sent Items; ItemCount=388\n2026-03-10T21:05:36Z,jpike,jpike,UserLoggedIn,Success,198.51.100.77,\"Mozilla/5.0 (iPhone; CPU iPhone OS 17_3 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148\",s-58c011,AuthMethod=Password; MFA=NotRegistered\n2026-03-11T12:58:33Z,jpike,jpike,UserLoggedIn,Success,198.51.100.77,\"Mozilla/5.0 (iPhone; CPU iPhone OS 17_3 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148\",s-58c7f2,AuthMethod=Password; MFA=NotRegistered\n2026-03-11T13:09:44Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-1301b5,AuthMethod=Password; MFA=NotRegistered\n2026-03-11T13:12:26Z,mbeaulieu,mbeaulieu,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-3c1d90,AuthMethod=Password+AuthenticatorApp; MFA=Satisfied\n2026-03-11T13:21:05Z,jpike,jpike,Send,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-1301b5,To=orders@castellanhardwood.example; Subject=PO 7751 - hard maple 5/4; Attachments=PO-7751.pdf\n2026-03-11T13:47:20Z,jpike,jpike,UserLoggedIn,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f9b04,AuthMethod=Password; MFA=NotRegistered\n2026-03-11T13:52:44Z,jpike,jpike,Send,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f9b04,To=mbeaulieu@orrinvalley.example; Subject=Castellan banking update - vendor change for Friday run; Attachments=Castellan_Bank_Letter_2026.pdf\n2026-03-11T13:53:05Z,jpike,jpike,MoveToDeletedItems,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f9b04,Folder=Sent Items; Subject=Castellan banking update - vendor change for Friday run\n2026-03-11T13:53:19Z,jpike,jpike,SoftDelete,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f9b04,Folder=Deleted Items; Subject=Castellan banking update - vendor change for Friday run\n2026-03-11T14:02:51Z,mbeaulieu,mbeaulieu,MailItemsAccessed,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-3c1d90,AccessType=Bind; Folder=Inbox; Subject=Castellan banking update - vendor change for Friday run\n2026-03-11T15:06:12Z,mbeaulieu,mbeaulieu,Send,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-3c1d90,To=jpike@orrinvalley.example; Subject=RE: Castellan banking update - vendor change for Friday run\n2026-03-11T15:20:31Z,jpike,jpike,MailItemsAccessed,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f9b04,AccessType=Bind; Folder=RSS Subscriptions; Subject=RE: Castellan banking update - vendor change for Friday run\n2026-03-12T13:01:17Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-1377e3,AuthMethod=Password; MFA=NotRegistered\n2026-03-12T16:44:30Z,jpike,jpike,UserLoggedIn,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-80a2d5,AuthMethod=Password; MFA=NotRegistered\n2026-03-12T16:45:02Z,jpike,jpike,MailItemsAccessed,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-80a2d5,AccessType=Bind; Folder=RSS Subscriptions; Subject=[EXTERNAL] Reminder: CHS-20417 due 13 March\n2026-03-12T16:45:40Z,jpike,jpike,MailItemsAccessed,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-80a2d5,AccessType=Bind; Folder=RSS Subscriptions; Subject=[EXTERNAL] RE: Invoice CHS-20417 - updated remittance details\n2026-03-13T12:55:48Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-13a5f1,AuthMethod=Password; MFA=NotRegistered\n2026-03-16T13:04:22Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-13d2c8,AuthMethod=Password; MFA=NotRegistered\n2026-03-16T19:32:08Z,itadmin,jpike,ResetPassword,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-a1e0f3,Target=jpike; ForceChangeAtNextLogin=True\n2026-03-16T19:32:40Z,itadmin,jpike,RevokeSessions,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-a1e0f3,Target=jpike\n2026-03-16T19:40:55Z,itadmin,jpike,Remove-InboxRule,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-a1e0f3,Mailbox=jpike; Identity=.\n2026-03-16T23:14:07Z,jpike,jpike,UserLoginFailed,InvalidPassword,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,,AuthMethod=Password\n2026-03-17T11:48:31Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-14f7b2,AuthMethod=Password+AuthenticatorApp; MFA=Satisfied\n",
"logs/web-proxy.log": "# SYNTHETIC TRAINING DATA. Fictional organizations, people, domains (RFC 2606 .example) and IPs (RFC 5737).\n# Orrin Valley Millworks web proxy export. Client OVM-WS-AP02 (Jordan Pike's workstation), 2026-03-10 and 2026-03-11.\n# Timestamps: appliance local time. No UTC offset is recorded in this export.\n# TLS inspection: enabled for this client; full URLs are logged.\n# Fields: date time client user method url status bytes category action\n2026-03-10 09:05:02 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/login 200 14022 Business/Email allowed\n2026-03-10 09:05:13 OVM-WS-AP02 jpike POST https://mail.orrinvalley.example/login 302 611 Business/Email allowed\n2026-03-10 09:05:14 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/inbox 200 88410 Business/Email allowed\n2026-03-10 09:31:40 OVM-WS-AP02 jpike GET https://erp.orrinvalley.example/ap/payment-runs 200 40317 Business/Internal allowed\n2026-03-10 09:58:12 OVM-WS-AP02 jpike GET https://www.weatherdesk.example/forecast 200 120544 News/Weather allowed\n2026-03-10 10:02:50 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/api/notifications 200 902 Business/Email allowed\n2026-03-10 10:18:44 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/inbox 200 90117 Business/Email allowed\n2026-03-10 10:18:51 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/read?id=AAMkAGQ5ZTc0 200 23875 Business/Email allowed\n2026-03-10 10:19:07 OVM-WS-AP02 jpike GET https://cstl-docshare.example/view/r?u=b3JyaW52YWxsZXkuZXhhbXBsZQ%3D%3D&t=anBpa2U%3D 200 5822 Newly-Registered-Domain allowed\n2026-03-10 10:19:08 OVM-WS-AP02 jpike GET https://cstl-docshare.example/assets/js/vw-portal.min.js 200 31740 Newly-Registered-Domain allowed\n2026-03-10 10:19:08 OVM-WS-AP02 jpike GET https://cstl-docshare.example/assets/css/vw.css 200 4410 Newly-Registered-Domain allowed\n2026-03-10 10:19:08 OVM-WS-AP02 jpike GET https://castellanhardwood.example/img/logo.png 200 18204 Business/Industrial allowed\n2026-03-10 10:19:09 OVM-WS-AP02 jpike GET https://cstl-docshare.example/favicon.ico 200 1150 Newly-Registered-Domain allowed\n2026-03-10 10:20:41 OVM-WS-AP02 jpike POST https://cstl-docshare.example/view/auth 302 388 Newly-Registered-Domain allowed\n2026-03-10 10:20:42 OVM-WS-AP02 jpike GET https://castellanhardwood.example/ 200 52310 Business/Industrial allowed\n2026-03-10 10:20:42 OVM-WS-AP02 jpike GET https://castellanhardwood.example/css/site.css?v=2025.3 200 9120 Business/Industrial allowed\n2026-03-10 10:21:30 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/inbox 200 90544 Business/Email allowed\n2026-03-10 11:02:17 OVM-WS-AP02 jpike GET https://www.sawdustjournal.example/march 200 77310 News/Trade allowed\n2026-03-11 09:09:30 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/login 200 14022 Business/Email allowed\n2026-03-11 09:09:43 OVM-WS-AP02 jpike POST https://mail.orrinvalley.example/login 302 611 Business/Email allowed\n2026-03-11 09:09:44 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/inbox 200 87102 Business/Email allowed\n2026-03-11 09:20:18 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/compose 200 12650 Business/Email allowed\n2026-03-11 09:21:05 OVM-WS-AP02 jpike POST https://mail.orrinvalley.example/mail/send 200 1204 Business/Email allowed\n2026-03-11 09:30:12 OVM-WS-AP02 jpike GET https://erp.orrinvalley.example/ap/invoices 200 51288 Business/Internal allowed\n2026-03-11 09:48:55 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/api/notifications 200 902 Business/Email allowed\n2026-03-11 09:52:30 OVM-WS-AP02 jpike GET https://erp.orrinvalley.example/ap/invoices?page=2 200 49870 Business/Internal allowed\n2026-03-11 09:55:10 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/api/notifications 200 902 Business/Email allowed\n2026-03-11 10:01:44 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/inbox 200 88003 Business/Email allowed\n2026-03-11 10:14:29 OVM-WS-AP02 jpike GET https://erp.orrinvalley.example/ap/payment-runs 200 40999 Business/Internal allowed\n"
}

In [ ]:
import hashlib                     # SHA-256: a impressão digital de cada arquivo
import urllib.request              # baixar sem instalar nada (requests não é stdlib)

RAW = "https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-7-capstone-career/case-packet/"   # endereço "cru" dos arquivos no GitHub

def baixar(nome, timeout=10):
    """Plano A: baixa o arquivo do GitHub. Devolve o texto ou None se falhar."""
    try:
        with urllib.request.urlopen(RAW + nome, timeout=timeout) as resp:   # timeout: não travar o Colab
            return resp.read().decode("utf-8")                              # bytes -> texto
    except Exception as erro:                                               # sem rede, 404, proxy...
        print(f"   (rede indisponível para {nome}: {type(erro).__name__}) -> uso a cópia embutida")
        return None

def sha(texto):
    """SHA-256 de um texto em UTF-8 (o mesmo que sha256sum daria no arquivo)."""
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

PACOTE = {}                                   # nome do arquivo -> texto que vamos analisar
for nome, copia in EMBUTIDO.items():
    online = baixar(nome)
    if online is None:                         # plano B
        PACOTE[nome] = copia
        origem = "embutido"
    else:                                      # plano A deu certo: confere com a cópia
        PACOTE[nome] = online
        origem = "GitHub" if sha(online) == sha(copia) else "GitHub (DIFERENTE da cópia embutida!)"
    print(f"{nome:32} {sha(PACOTE[nome])[:16]}…  [{origem}]")

## 0. Fuso com regras de horário de verão (e um plano B)

O original usa `zoneinfo.ZoneInfo("America/Toronto")`. No Colab e no Linux funciona. No Windows, o Python às vezes não traz a base de fusos (o pacote `tzdata`). Para o notebook não quebrar, a célula abaixo tenta o `zoneinfo` e, se falhar, usa uma **regra escrita à mão** para 2026: o Leste dos EUA/Canadá é UTC-5 até 8/mar/2026 02:00 local, e UTC-4 depois (até novembro).

In [ ]:
from datetime import datetime, timedelta, timezone

try:
    from zoneinfo import ZoneInfo
    LESTE = ZoneInfo("America/Toronto")              # a base oficial de fusos (IANA)
    def para_utc_leste(local_ingenuo):
        return local_ingenuo.replace(tzinfo=LESTE).astimezone(timezone.utc)
    print("zoneinfo OK: usando a base IANA")
except Exception:                                     # ZoneInfoNotFoundError, ImportError...
    INICIO_DST_2026 = datetime(2026, 3, 8, 2, 0)      # 2º domingo de março, 02:00 local
    FIM_DST_2026 = datetime(2026, 11, 1, 2, 0)        # 1º domingo de novembro, 02:00 local
    def para_utc_leste(local_ingenuo):
        verao = INICIO_DST_2026 <= local_ingenuo < FIM_DST_2026
        return (local_ingenuo + timedelta(hours=4 if verao else 5)).replace(tzinfo=timezone.utc)
    print("zoneinfo indisponível: usando a regra manual de 2026 (só vale para este caso!)")

# Teste rápido: um horário antes e um depois de 8 de março
for t in ("2026-03-03 09:47:00", "2026-03-10 09:05:13"):
    print(t, "local ->", para_utc_leste(datetime.fromisoformat(t)).strftime("%Y-%m-%d %H:%M:%SZ"))

## 1. Lendo os três logs

Os CSVs começam com linhas `#` de comentário; o original filtra essas linhas antes do `DictReader`. Fazemos o mesmo.

In [ ]:
import csv, io

def linhas_csv(texto):
    """CSV sem as linhas de comentário '#' (mesma ideia do rows() do original)."""
    return list(csv.DictReader(io.StringIO("\n".join(l for l in texto.splitlines() if not l.startswith("#")))))

P7 = linhas_csv(PACOTE["logs/mail-trace.csv"])
P8 = linhas_csv(PACOTE["logs/mailbox-audit.csv"])
P9 = []
for linha in PACOTE["logs/web-proxy.log"].splitlines():
    if linha.startswith("#") or not linha.strip():
        continue
    d, t, host, user, method, url, status, size, cat, action = linha.split()   # 10 campos, como no original
    P9.append({"local": datetime.strptime(f"{d} {t}", "%Y-%m-%d %H:%M:%S"),
               "method": method, "url": url, "status": status, "cat": cat, "user": user, "host": host})
print(f"P7: {len(P7)} linhas · P8: {len(P8)} linhas · P9: {len(P9)} linhas · total {len(P7)+len(P8)+len(P9)}")

## 2. As âncoras

Procuramos todo `POST .../login` no P9 e todo `UserLoggedIn` do `jpike` a partir do IP do escritório (`192.0.2.10`) no P8. Um par com diferença de **um segundo mais um número inteiro de horas** é o mesmo evento visto por dois relógios.

In [ ]:
def utc(s):
    """'2026-03-10T13:05:14Z' -> datetime UTC."""
    return datetime.fromisoformat(s.replace("Z", "+00:00"))

posts = [e for e in P9 if e["method"] == "POST" and e["url"].endswith("/login")]
logins_escritorio = [r for r in P8 if r["operation"] == "UserLoggedIn"
                     and r["actor"] == "jpike" and r["client_ip"] == "192.0.2.10"]

ANCORAS = []
for p in posts:
    for r in logins_escritorio:
        u = utc(r["timestamp_utc"]).replace(tzinfo=None)      # sem fuso, para subtrair do local
        diff = (u - p["local"]).total_seconds()
        horas, resto = divmod(diff, 3600)
        if 0 <= resto <= 5:                                    # até 5 s de latência
            ANCORAS.append((p["local"], r["timestamp_utc"], r["session_id"], int(horas)))

for local, z, sess, h in ANCORAS:
    print(f"P9 {local}  ×  P8 {z} ({sess})  ->  UTC = local + {h} h  (ou seja, UTC-{h})")
dias = {a[0].date() for a in ANCORAS}
print(f"\n{len(ANCORAS)} âncoras em {len(dias)} dias diferentes; deslocamentos: {sorted({a[3] for a in ANCORAS})}")

Duas âncoras, dois dias (10 e 11/mar), o mesmo deslocamento: **UTC-4**. O palpite da TI estava certo, e agora isso é evidência, não palpite.

### 2.1 A armadilha de 3 de março

O phishing cita M1 como *"Tue, Mar 3, 2026 at 9:47 AM"*. O gateway registrou M1 às 14:47:04 UTC.

In [ ]:
m1 = next(r for r in P7 if "Invoice CHS-20417" == r["subject"].replace("[EXTERNAL] ", ""))
m1_utc = utc(m1["received_utc"])
for horas in (4, 5):
    local = (m1_utc - timedelta(hours=horas)).strftime("%H:%M")
    print(f"14:47 UTC com UTC-{horas} = {local} local  -> {'bate com 9:47 AM' if local == '09:47' else 'NÃO bate'}")
print("Com a regra de horário de verão:",
      "9:47 local de 3/mar =", para_utc_leste(datetime(2026, 3, 3, 9, 47)).strftime("%H:%MZ"))

Quem aplica -4 em tudo encontra uma diferença falsa de uma hora e pode concluir que a citação foi adulterada. Ela não foi.

## 3. O script do original: 80 eventos em UTC

In [ ]:
out = []
for r in P7:
    out.append({"datetime": r["received_utc"].replace("Z", "+00:00"),
                "timestamp_desc": "Message received (gateway)",
                "message": f'{r["direction"]} {r["sender"]} -> {r["recipients"]}: {r["subject"]}',
                "source": "P7", "user": "", "client": r["sender_ip"], "session_id": ""})
for r in P8:
    out.append({"datetime": r["timestamp_utc"].replace("Z", "+00:00"),
                "timestamp_desc": "Audit event",
                "message": f'{r["operation"]} ({r["result"]}) by {r["actor"]} on {r["target_mailbox"]}: {r["details"]}',
                "source": "P8", "user": r["actor"], "client": r["client_ip"], "session_id": r["session_id"]})
for e in P9:
    out.append({"datetime": para_utc_leste(e["local"]).isoformat(),     # conversão com regra de DST
                "timestamp_desc": "Proxy request (converted from local time)",
                "message": f'{e["method"]} {e["url"]} -> {e["status"]} [{e["cat"]}]',
                "source": "P9", "user": e["user"], "client": e["host"], "session_id": ""})

# ➕ As três linhas do P2, acrescentadas "à mão" como o original pede
from email import policy
from email.parser import Parser
msg = Parser(policy=policy.default).parsestr(PACOTE["02-suspicious-email.eml"])
from email.utils import parsedate_to_datetime
for i, rec in enumerate(reversed(msg.get_all("Received")), 1):
    quando = parsedate_to_datetime(str(rec).split(";")[-1].strip())
    out.append({"datetime": quando.astimezone(timezone.utc).isoformat(), "timestamp_desc": f"Received header hop {i} (P2)",
                "message": " ".join(str(rec).split())[:120], "source": "P2", "user": "", "client": "", "session_id": ""})

# Normaliza o formato para ordenar texto com segurança (todos em +00:00)
for e in out:
    e["datetime"] = datetime.fromisoformat(e["datetime"]).astimezone(timezone.utc).isoformat()
out.sort(key=lambda e: e["datetime"])
print(f"{len(out)} eventos (80 dos logs + 3 do P2)")

with open("timeline_utc.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(out[0].keys()))
    w.writeheader(); w.writerows(out)

# Confira 3 linhas do proxy à mão, como o original manda
for e in [x for x in out if x["source"] == "P9"][8:11]:
    print(e["datetime"], e["message"][:80])

## 4. 🎮 Quiz das sessões

Para cada sessão (e para os logins com falha), o notebook mostra IP, user agent, horário e o que ela fez. Você responde: **J** (Jordan), **A** (atacante), **M** (Maren), **T** (TI) ou **D** (desconhecido). Depois vê a atribuição sugerida e os motivos **além do IP**.

In [ ]:
from collections import OrderedDict

sessoes = OrderedDict()
for r in P8:
    chave = r["session_id"] or f"(falha {r['client_ip']})"            # falhas não têm sessão
    s = sessoes.setdefault(chave, {"ip": r["client_ip"], "ua": r["user_agent"][:38], "ev": [], "ops": set(),
                                    "contas": set()})
    s["ev"].append(r["timestamp_utc"]); s["ops"].add(r["operation"]); s["contas"].add(r["target_mailbox"])

ATRIB = {   # atribuição sugerida e motivo além do IP (o fluxograma do original)
    "192.0.2.10": ("J", "IP do escritório (L8) + Chrome/Windows de sempre + navegação contínua no P9"),
    "198.51.100.77": ("J", "Operadora móvel CGNAT (L8) + iPhone + já aparece em 9/mar, ANTES do phishing; confirmar com o Jordan"),
    "198.51.100.23": ("A", "VPS (L8) + Linux Firefox nunca visto para o jpike + mesmo IP do submissor do phishing (P2); a 1ª sessão veio 10 min após o POST no phishing (P9)"),
    "203.0.113.200": ("D", "Spray em 6 contas, tudo InvalidPassword, IP em lista pública de spray: sem ligação demonstrada"),
}
MODO_JOGO = True
acertos = total = 0
for sid, s in sessoes.items():
    contas = ",".join(sorted(s["contas"]))
    sugestao, motivo = ATRIB.get(s["ip"], ("D", "sem padrão conhecido"))
    if "itadmin" in s["ev"] or "ResetPassword" in s["ops"]:
        sugestao, motivo = "T", "Operações de administrador (ResetPassword, RevokeSessions, Remove-InboxRule) do escritório"
    if "mbeaulieu" in contas and s["ip"] == "192.0.2.10":
        sugestao, motivo = "M", "Conta mbeaulieu com MFA satisfeito (AuthenticatorApp) a partir do escritório"
    print(f"\n{sid:22} {s['ip']:15} {s['ua']:38} {s['ev'][0][5:16]}..{s['ev'][-1][11:16]}  {sorted(s['ops'])}")
    if MODO_JOGO:
        r = input("   J, A, M, T ou D? ").strip().upper()[:1]
        total += 1; acertos += (r == sugestao)
    print(f"   sugerido: {sugestao} — {motivo}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{total} batendo com a sugestão")

> ⚠️ A célula agrupa os logins com falha **por IP**: `(falha 203.0.113.200)` junta o spray em seis contas, e `(falha 198.51.100.23)` é a tentativa do atacante em 16/mar, **depois** da troca de senha. Essa última é a evidência de que a troca funcionou.

## 5. Evidência negativa: o envio das 13:52:44 de 11/mar

Se o Jordan tivesse mandado a carta bancária pelo navegador do escritório, o proxy mostraria um `POST /mail/send` perto das 13:52 UTC. Vamos procurar.

In [ ]:
envios_p8 = [r for r in P8 if r["operation"] == "Send" and r["actor"] == "jpike"]
envios_p9 = [para_utc_leste(e["local"]) for e in P9 if e["method"] == "POST" and e["url"].endswith("/mail/send")]

for r in envios_p8:
    t = utc(r["timestamp_utc"])
    perto = [x for x in envios_p9 if abs((x - t).total_seconds()) <= 120]   # janela de 2 min
    print(f"P8 Send {r['timestamp_utc']} de {r['client_ip']:15} sessão {r['session_id']}: "
          f"{'tem' if perto else 'NÃO tem'} POST /mail/send no P9  | {r['details'][:60]}")

O envio das 13:21 (pedido legítimo à Castellan) aparece nos dois logs. O das 13:52 (a carta falsa) **não aparece no proxy**, e saiu de `198.51.100.23`. A ausência enfraquece a rival "o Jordan mandou a carta e está encobrindo".

## 6. ➕ Os três relógios da Operação Isca

In [ ]:
# Dados sintéticos da Operação Isca deste dia (plano B embutido).
ISCA_EMBUTIDO = {
"ouvidoria.csv": "# FICTÍCIO — Operação Isca (investigador-cyber-90-dias). Nenhuma pessoa, conta, número ou domínio real.\n# Horario de Brasilia (BRT, UTC-3), digitado pelo atendente.\nregistrado_brt,protocolo,resumo\n2026-10-05 07:40,OUV-0001,cooperado recebeu SMS 'Pix bloqueado' e nao clicou\n2026-10-05 07:52,OUV-0004,cooperado V1 clicou e digitou a senha do internet banking\n2026-10-05 08:25,OUV-0011,cooperado V2 clicou e digitou senha; depois viu Pix que nao fez\n2026-10-05 08:57,OUV-0023,cooperado V3 relata Pix nao reconhecido\n2026-10-05 09:10,OUV-0037,37a reclamacao do dia sobre o mesmo SMS\n",
"ib-acessos.csv": "# FICTÍCIO — Operação Isca (investigador-cyber-90-dias). Nenhuma pessoa, conta, número ou domínio real.\n# Internet banking da Serra Azul. Horario em UTC (NTP).\nts_utc,cooperado,ip,dispositivo,evento,detalhe\n2026-10-05T10:49:02Z,V1,198.51.100.4,Android,login,celular habitual do cooperado\n2026-10-05T11:03:40Z,V1,203.0.113.150,Linux Firefox,login,dispositivo novo\n2026-10-05T11:05:12Z,V1,203.0.113.150,Linux Firefox,pix_enviado,PIXFICT-L01 (chave ficticia) R$ 1.950\n2026-10-05T11:12:30Z,V2,203.0.113.150,Linux Firefox,login,dispositivo novo\n2026-10-05T11:14:05Z,V2,203.0.113.150,Linux Firefox,pix_enviado,PIXFICT-L01 (chave ficticia) R$ 2.400\n2026-10-05T11:15:44Z,V2,203.0.113.150,Linux Firefox,pix_enviado,PIXFICT-L02 (chave ficticia) R$ 1.100\n2026-10-05T11:41:19Z,V3,203.0.113.150,Linux Firefox,login,dispositivo novo\n2026-10-05T11:43:02Z,V3,203.0.113.150,Linux Firefox,pix_enviado,PIXFICT-L03 (chave ficticia) R$ 3.300\n2026-10-05T14:20:00Z,SISTEMA,10.20.0.5,servidor,bloqueio_preventivo,login de dispositivo novo exige confirmacao na agencia\n2026-10-06T12:00:01Z,SISTEMA,10.20.0.5,servidor,teste_integracao,id=TI-0610 enviado ao sistema de contestacao\n",
"pix-contestacoes.log": "# FICTÍCIO — Operação Isca (investigador-cyber-90-dias). Nenhuma pessoa, conta, número ou domínio real.\n# Sistema de contestacao de Pix. Horario local do appliance. Nenhum fuso anotado.\n2026-10-05 09:31:15 V1 contestacao_aberta pix_ref=PIXFICT-E2E-0001 motivo=fraude\n2026-10-05 09:31:16 V1 med_solicitado pix_ref=PIXFICT-E2E-0001\n2026-10-05 09:47:10 V2 contestacao_aberta pix_ref=PIXFICT-E2E-0002 motivo=fraude\n2026-10-05 09:47:12 V2 contestacao_aberta pix_ref=PIXFICT-E2E-0003 motivo=fraude\n2026-10-05 10:12:05 V3 contestacao_aberta pix_ref=PIXFICT-E2E-0004 motivo=fraude\n2026-10-05 11:20:01 SISTEMA regra_recebida origem=internet_banking regra=bloqueio_preventivo\n2026-10-06 09:00:02 SISTEMA teste_integracao origem=internet_banking id=TI-0610\n"
}

In [ ]:
import urllib.request

RAW_ISCA = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-86/"     # pasta caso-continuo/dados/dia-86 no GitHub

ISCA = {}
for nome, copia in ISCA_EMBUTIDO.items():
    try:                                                     # plano A: versão publicada no repo
        with urllib.request.urlopen(RAW_ISCA + nome, timeout=10) as r:
            ISCA[nome] = r.read().decode("utf-8")
        print(f"{nome}: baixado do GitHub")
    except Exception:                                        # plano B: cópia embutida
        ISCA[nome] = copia
        print(f"{nome}: cópia embutida")

In [ ]:
IB = linhas_csv(ISCA["ib-acessos.csv"])
OUV = linhas_csv(ISCA["ouvidoria.csv"])
PIX = []
for l in ISCA["pix-contestacoes.log"].splitlines():
    if l.startswith("#") or not l.strip():
        continue
    d, t, quem, evento, *resto = l.split()
    PIX.append({"local": datetime.fromisoformat(f"{d} {t}"), "quem": quem, "evento": evento, "resto": " ".join(resto)})

# Âncoras: eventos do SISTEMA que aparecem nos dois lados
ancoras_isca = [
    (next(p for p in PIX if p["evento"] == "regra_recebida")["local"],
     utc(next(r for r in IB if r["evento"] == "bloqueio_preventivo")["ts_utc"]).replace(tzinfo=None)),
    (next(p for p in PIX if p["evento"] == "teste_integracao")["local"],
     utc(next(r for r in IB if r["evento"] == "teste_integracao")["ts_utc"]).replace(tzinfo=None)),
]
for local, u in ancoras_isca:
    print(f"contestação {local}  ×  IB {u}Z  ->  diferença {round((u - local).total_seconds() / 3600, 2)} h")

In [ ]:
#@title Deslocamento do sistema de contestação { display-mode: "form" }
horas_somar_para_utc = 3  #@param {type:"integer"}

print("Linha do tempo do Isca em UTC (fonte entre colchetes):\n")
ev = []
for r in OUV:   # ouvidoria em BRT (UTC-3, Brasil sem horário de verão desde 2019)
    ev.append((datetime.fromisoformat(r["registrado_brt"]) + timedelta(hours=3), "OUV", f'{r["protocolo"]} {r["resumo"]}'))
for r in IB:
    ev.append((utc(r["ts_utc"]).replace(tzinfo=None), "IB ", f'{r["cooperado"]} {r["evento"]} {r["ip"]} {r["detalhe"]}'))
for p in PIX:
    ev.append((p["local"] + timedelta(hours=horas_somar_para_utc), "PIX", f'{p["quem"]} {p["evento"]} {p["resto"]}'))
for t, f, m in sorted(ev):
    if t.date().day == 5:
        print(f"{t:%H:%M:%S}Z [{f}] {m[:85]}")

# Checagem de sanidade: alguma contestação antes do Pix correspondente?
for v in ("V1", "V2", "V3"):
    pix = min(t for t, f, m in ev if f == "IB " and m.startswith(v) and "pix_enviado" in m)
    cont = min(t for t, f, m in ev if f == "PIX" and m.startswith(v))
    print(f"\n{v}: Pix {pix:%H:%M}Z, contestação {cont:%H:%M}Z ->",
          "OK" if cont > pix else "⚠️ contestação ANTES do Pix: o deslocamento está errado")

Rode a célula de novo com `horas_somar_para_utc = 0` (o palpite da TI, "é UTC") e veja os três avisos aparecerem. É essa a armadilha da abertura.

## 7. 📝 Registro de decisões e limites

In [ ]:
#@title Notas do Dia 86 { display-mode: "form" }
ancoras_capstone = "P9 2026-03-10 09:05:13 × P8 13:05:14Z (s-12f4a8); P9 2026-03-11 09:09:43 × P8 13:09:44Z (s-1301b5): UTC-4"  #@param {type:"string"}
ancoras_isca = "Contestação 05/10 11:20:01 × IB 14:20:00Z; 06/10 09:00:02 × IB 12:00:01Z: horário local = UTC-3 (a TI achava UTC)"  #@param {type:"string"}
limites = "P8 só jpike e mbeaulieu, de 9/mar a 17/mar 12:00Z; sem ap@; regra apagada pela TI antes da exportação (16/mar 19:40Z)"  #@param {type:"string"}

from pathlib import Path
Path("notas").mkdir(exist_ok=True)
texto = f"# Dia 86\n\n## Âncoras\n- {ancoras_capstone}\n- {ancoras_isca}\n\n## Limites\n{limites}\n"
Path("notas/decisoes-dia86.md").write_text(texto, encoding="utf-8")
print(texto)
try:
    from google.colab import files
    files.download("timeline_utc.csv"); files.download("notas/decisoes-dia86.md")
except ImportError:
    print("(Fora do Colab: timeline_utc.csv e notas/decisoes-dia86.md ficaram na pasta atual)")

## ✅ Antes de fechar o Dia 86

- [ ] Duas âncoras em dias diferentes para o proxy (capstone) e para o sistema de contestação (Isca)
- [ ] A armadilha de 3 de março explicada com as suas palavras
- [ ] `timeline_utc.csv` com 80 eventos dos logs (+3 do P2), três conversões conferidas à mão
- [ ] Tabela de sessões com motivo além do IP em todas
- [ ] Evidência negativa do envio de 11/mar registrada, com a hipótese que ela enfraquece

**Amanhã:** Dia 87: análise de fraude e engenharia social. Cada mensagem do atacante foi escrita para derrotar um controle específico. Quais?

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>